# 🗂️ Notebook 4 — API Center Registration and Runtime Discovery Sync

This notebook registers the deployed specialist agents in **Azure API Center** as the governance catalog,  
then syncs that catalog to a **runtime discovery snapshot** used by the orchestrator for dynamic routing.

## Why this sequence matters
```
Specialists deployed (Notebook 3)
        │
        ▼ (register with metadata: capabilities, persona, trust, risk)
  Azure API Center  ←── Governance catalog / source of truth
        │
        ▼ (sync: pull active entries, validate schema, materialize snapshot)
  registry/runtime-snapshot.json  ←── Orchestrator reads this at startup
        │
        ▼
  pf-orchestrator uses snapshot to route requests dynamically
```

## What this notebook does
1. Verifies the API Center resource in the hub resource group
2. Registers each specialist as an API with governance metadata
3. Validates that all required metadata fields are present  
4. Materializes the runtime snapshot JSON from API Center data
5. Validates the snapshot checksum and saves it for the orchestrator

In [ ]:
import sys, json, pathlib, hashlib, datetime, subprocess
sys.path.insert(0, str(pathlib.Path("../../shared").resolve()))
import utils

def run(cmd, ok="", fail=""):
    return utils.run(cmd, ok, fail)

def azd_get(key: str) -> str:
    p = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Missing azd env value [{key}]: {(p.stderr or p.stdout).strip()}")
    return (p.stdout or "").strip()

def azd_get_optional(key: str, default: str = "") -> str:
    p = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if p.returncode != 0:
        return default
    return (p.stdout or "").strip() or default

SUB_ID = azd_get("AZURE_SUBSCRIPTION_ID")
hub_rg = azd_get("AZURE_RESOURCE_GROUP")
account = azd_get("SPOKE_AI_FOUNDRY_ACCOUNT_NAME")
project = azd_get("SPOKE_AI_FOUNDRY_PROJECT_NAME")
FOUNDRY_EP = azd_get_optional("FOUNDRY_PROJECT_ENDPOINT", f"https://{account}.services.ai.azure.com/api/projects/{project}")

deployed_raw = azd_get_optional("PF_DEPLOYED_SPECIALISTS", "[]")
try:
    deployed = json.loads(deployed_raw)
except Exception:
    deployed = []

utils.print_info(f"Hub RG:             {hub_rg}")
utils.print_info(f"Deployed specialists: {deployed}")

if not deployed:
    raise RuntimeError("No deployed specialists found in env. Run Notebook 3 first.")

### 1️⃣ Find API Center resource in hub resource group

In [ ]:
apic_out = run(
    f"az resource list -g {hub_rg} "
    f"--resource-type Microsoft.ApiCenter/services -o json",
    "API Center query OK", "API Center query failed"
)

if not apic_out.success or not apic_out.json_data:
    utils.print_warning(
        "No API Center resource found in hub RG. The Citadel deployment may not have included it.\n"
        "Skipping API Center registration — using local registry metadata as fallback."
    )
    apic_name  = None
    apic_found = False
else:
    apic_name  = apic_out.json_data[0]["name"]
    apic_found = True
    utils.print_ok(f"API Center found: {apic_name}")

### 2️⃣ Register deployed specialists in API Center

Each agent is registered as an API with governance metadata embedded in custom properties.  
The metadata schema follows the `registry/agents-metadata.json` contract.

In [ ]:
AGENT_GOVERNANCE_METADATA = {
    "pf-contextualizer": {
        "capabilities": ["intent_extraction","entity_extraction","query_completion","risk_classification"],
        "allowed_personas": ["external_customer","internal_scientist"],
        "trust_level": "verified", "risk_tiers_supported": ["low","elevated"],
        "execution_order": "first", "domain": "product-finder",
    },
    "pf-product-intelligence": {
        "capabilities": ["product_recommendation","product_search","catalog_retrieval"],
        "allowed_personas": ["external_customer","internal_scientist"],
        "trust_level": "verified", "risk_tiers_supported": ["low","elevated"],
        "execution_order": "after_contextualizer", "domain": "product-finder",
    },
    "pf-compatibility": {
        "capabilities": ["compatibility_check","risk_assessment","confidence_scoring"],
        "allowed_personas": ["external_customer","internal_scientist"],
        "trust_level": "verified", "risk_tiers_supported": ["elevated"],
        "min_confidence_threshold": 0.90,
        "execution_order": "parallel_with_product_intelligence", "domain": "product-finder",
    },
    "pf-aligner": {
        "capabilities": ["intent_alignment","response_validation","formatting"],
        "allowed_personas": ["external_customer","internal_scientist"],
        "trust_level": "verified", "risk_tiers_supported": ["low","elevated"],
        "execution_order": "last", "domain": "product-finder",
    },
    "pf-sample-request": {
        "capabilities": ["sample_request_processing","authentication_verification"],
        "allowed_personas": ["external_customer"],
        "trust_level": "verified", "risk_tiers_supported": ["low"],
        "execution_order": "standalone", "domain": "product-finder",
    },
}

registered_entries = []
for agent_name in deployed:
    meta = AGENT_GOVERNANCE_METADATA.get(agent_name, {})
    if apic_found:
        # Register in API Center
        api_id = agent_name.replace("-", "")
        create_out = run(
            f"az apic api create -g {hub_rg} -n {apic_name} "
            f"--api-id {api_id} "
            f"--title \"{agent_name}\" "
            f"--type \"REST\" -o json",
            f"  Registered {agent_name} in API Center",
            f"  API Center registration failed for {agent_name} (may already exist)"
        )
        utils.print_info(f"  API Center: {agent_name} → {'OK' if create_out.success else 'already exists or skipped'}")

    # Build registry entry regardless (used for runtime snapshot)
    # gateway_path is the APIM route the orchestrator should call after discovery.
    entry = {
        "id":               agent_name,
        "agent_name":       agent_name,
        "display_name":     agent_name.replace("-", " ").title(),
        "status":           "active",
        "version":          "1.0.0",
        "domain":           "product-finder",
        "foundry_endpoint": FOUNDRY_EP,
        "gateway_path":     f"/product-finder/agents/{agent_name}/responses",
        **meta,
    }
    registered_entries.append(entry)
    utils.print_ok(f"  {agent_name}: registry entry built")

utils.print_info(f"\nTotal registered: {len(registered_entries)} agents")

In [ ]:
# ── 3️⃣  Materialize runtime discovery snapshot ─────────────────────────────
REQUIRED_FIELDS = {"id","agent_name","status","capabilities","allowed_personas",
                   "trust_level","risk_tiers_supported","domain"}

errors = []
for entry in registered_entries:
    missing = REQUIRED_FIELDS - set(entry.keys())
    if missing:
        errors.append(f"{entry['id']}: missing fields {sorted(missing)}")
if errors:
    raise RuntimeError("Registry schema validation FAILED:\n" + "\n".join(errors))
utils.print_ok("Registry schema validation passed")

snapshot = {
    "snapshot_id":      f"pf-snapshot-{datetime.datetime.utcnow().strftime('%Y%m%d%H%M%S')}",
    "generated_at":     datetime.datetime.utcnow().isoformat() + "Z",
    "schema_version":   "1.0",
    "environment":      "workshop",
    "agents":           registered_entries,
}
snapshot_str = json.dumps(snapshot, separators=(",", ":"), sort_keys=True)
snapshot["checksum"] = hashlib.sha256(snapshot_str.encode()).hexdigest()

snap_path = pathlib.Path("../registry/runtime-snapshot.json")
snap_path.write_text(json.dumps(snapshot, indent=2), encoding="utf-8")
utils.print_ok(f"Runtime snapshot written: {snap_path.resolve()}")
utils.print_info(f"  Snapshot ID:  {snapshot['snapshot_id']}")
utils.print_info(f"  Agent count:  {len(registered_entries)}")
utils.print_info(f"  Checksum:     {snapshot['checksum'][:16]}...")

# Active agent summary
print("\n── ACTIVE AGENTS IN RUNTIME SNAPSHOT ──────────────────────────────")
for entry in registered_entries:
    caps = ", ".join(entry.get("capabilities", []))
    personas = ", ".join(entry.get("allowed_personas", []))
    print(f"  ✅ {entry['agent_name']:35s}  personas: {personas}")
    print(f"      caps: {caps}")

# Persist snapshot metadata to azd env for downstream notebooks.
for k, v in {
    "PF_RUNTIME_SNAPSHOT_ID": snapshot["snapshot_id"],
    "PF_RUNTIME_SNAPSHOT_CHECKSUM": snapshot["checksum"],
}.items():
    p = subprocess.run(["azd", "env", "set", k, str(v)], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Failed to persist {k} to azd env: {(p.stderr or p.stdout).strip()}")
utils.print_ok("Persisted runtime snapshot metadata to azd env")

print()
utils.print_ok("✅ API Center registration and runtime sync COMPLETE. Proceed to Notebook 5.")